# Preprocessing CTFF

In [1]:
# Import
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl

from tqdm.notebook import tqdm
from util.system import get_data
from util.data import save_parquet_chunk, read_parquet_chunk

In [17]:
crsp = pd.read_csv(get_data() / 'crsp' / 'crsp_monthly_index_and_portfolios_on_sp500.csv')
print(crsp.columns)
print(crsp)

Index(['INDNO', 'YYYYMM', 'MthCalDt', 'MthTotRet', 'MthTotInd', 'MthPrcRet',
       'MthPrcInd', 'MthIncRet', 'MthIncInd', 'MthUsdCnt', 'MthUsdVal',
       'MthTotCnt', 'MthTotVal', 'MthEligCnt', 'MthWgtAmt', 'INDFAM',
       'IndFamType', 'IndNm', 'IndBegDt', 'IndEndDt', 'BaseLvl', 'BaseDt',
       'FreqAvail', 'WeightType', 'CntValType', 'PortNum'],
      dtype='object')
        INDNO  YYYYMM    MthCalDt  MthTotRet  MthTotInd  MthPrcRet  MthPrcInd  \
0     1000502  198911  1989-11-30        NaN        NaN   0.016541     345.99   
1     1000502  197903  1979-03-30        NaN        NaN   0.055152     101.59   
2     1000502  195711  1957-11-29        NaN        NaN   0.016074      41.72   
3     1000502  200007  2000-07-31        NaN        NaN  -0.016341    1430.83   
4     1000502  197311  1973-11-30        NaN        NaN  -0.113861      95.96   
...       ...     ...         ...        ...        ...        ...        ...   
1196  1000502  201111  2011-11-30        NaN        NaN  

In [21]:
crsp.sort_values("YYYYMM").dropna(subset="MthIncRet")

,INDNO,YYYYMM,MthCalDt,MthTotRet,MthTotInd,MthPrcRet,MthPrcInd,MthIncRet,MthIncInd,MthUsdCnt,...,IndFamType,IndNm,IndBegDt,IndEndDt,BaseLvl,BaseDt,FreqAvail,WeightType,CntValType,PortNum


### Filter CTFF

In [2]:
# CTFF List
ctff_list = pd.read_csv(get_data() / 'ctff' / 'ctff_list.csv')['features'].tolist()

# JKP List
jkp_list = pd.read_csv(get_data() / 'jkp' / 'jkp_153_list.csv')['characteristic'].tolist()

In [3]:
# Read JKP chunks
ctff_m = read_parquet_chunk(folder_path=get_data() / 'ctff' / 'ctff_m', file_pattern='ctff_m_*')
ctff_d = read_parquet_chunk(folder_path=get_data() / 'ctff' / 'ctff_d', file_pattern='ctff_d_*')

Chunk: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 864/864 [00:15<00:00, 54.43it/s]


In [4]:
# Set date
ctff_m["eom"] = ctff_m["eom"].str.replace("-", "", regex=False).astype(np.int32)
ctff_m = ctff_m[(ctff_m["eom"] >= 19520101) & (ctff_m["eom"] <= 20231231)]

In [5]:
# Select size
ctff_m = ctff_m.loc[ctff_m.size_grp.isin(['small', 'large', 'mega'])]

In [6]:
# Keep stock-months with >= 200 non-missing daily returns in past 252 trading days
ctff_d["date"] = pd.to_datetime(ctff_d["date"])
ctff_d = ctff_d.sort_values(["id", "date"])
ctff_d["is_ret"] = ctff_d["ret_exc"].notna().astype(np.int16)
ctff_d["count_252"] = ctff_d.groupby("id")["is_ret"].rolling(252, min_periods=252).sum().reset_index(level=0, drop=True)
ctff_d["eom"] = ctff_d["date"].dt.to_period("M").dt.to_timestamp("M")
month_end_count = ctff_d.groupby(["id", "eom"], sort=False).tail(1)[["id", "eom", "count_252"]]
month_end_count["eom"] = month_end_count["eom"].dt.strftime("%Y%m%d").astype(np.int32)

In [7]:
# Keep stock-months with >= 200 non-missing daily returns in past 252 trading days
ctff_m = ctff_m.merge(month_end_count, on=["id", "eom"], how="left")
print(f"ctff_m.shape before filter: {ctff_m.shape}")
ctff_m = ctff_m[ctff_m["count_252"] >= 200]
print(f"ctff_m.shape after filter: {ctff_m.shape}")

ctff_m.shape before filter: (1369181, 411)
ctff_m.shape after filter: (1353925, 411)


In [8]:
# Require at least 75 non-missing characteristics from jkp_list (roughly 50 percent)
ctff_m["numchar_nonmiss"] = ctff_m[jkp_list].notna().sum(axis=1)
print(f"ctff_m.shape before filter: {ctff_m.shape}")
ctff_m = ctff_m[ctff_m["numchar_nonmiss"] >= 75]
print(f"ctff_m.shape after filter: {ctff_m.shape}")

ctff_m.shape before filter: (1353925, 412)
ctff_m.shape after filter: (1353925, 412)


In [9]:
# Set index
ctff_m.set_index(['id', 'eom'], inplace=True)
ctff_m.sort_index(inplace=True)

In [10]:
# Keep only JKP charateristics
ctff_m = ctff_m[['ret_exc_lead1m', 'size_grp'] + jkp_list]

### Rank-Normalize

In [11]:
def rank_normalize(df, exclude_cols: list[str]) -> pl.DataFrame:
    """Rank-normalize numeric columns by eom using only finite values; impute non-finite/missing outputs to 0."""
    pdf = df.reset_index()
    cols = [c for c in pdf.columns if c not in (exclude_cols + ["id", "eom"])]
    pl_df = pl.from_pandas(pdf)

    for col in tqdm(cols, desc="Column"):
        clean_col = f"{col}_clean"
        rank_col = f"{col}_rank"

        # Keep only finite values for ranking; null/NaN/±inf become null.
        pl_df = pl_df.with_columns(
            pl.when(pl.col(col).is_finite())
            .then(pl.col(col))
            .otherwise(None)
            .alias(clean_col)
        )

        # Rank within each eom on finite values only.
        pl_df = pl_df.with_columns(
            pl.col(clean_col)
            .rank(method="average")
            .over("eom")
            .alias(rank_col)
        )

        # Scale realized ranks to [-0.5, 0.5]; degenerate groups -> 0.
        r_min = pl.col(rank_col).min().over("eom")
        r_max = pl.col(rank_col).max().over("eom")
        denom = r_max - r_min

        pl_df = (
            pl_df.with_columns(
                pl.when(denom == 0)
                .then(0.0)
                .otherwise(((pl.col(rank_col) - r_min) / denom) - 0.5)
                .fill_null(0.0)
                .fill_nan(0.0)
                .alias(col)
            )
            .drop([clean_col, rank_col])
        )

    keep_cols = ["id", "eom"] + cols + [c for c in exclude_cols if c in pl_df.columns]
    out = pl_df.select(keep_cols).to_pandas()
    out.set_index(["id", "eom"], inplace=True)
    out.sort_index(inplace=True)
    return out

In [12]:
# Size
ctff_m_all = ctff_m.copy().drop(columns='size_grp', axis=1)
ctff_m_mega = ctff_m.loc[ctff_m['size_grp'] == 'mega'].copy().drop(columns='size_grp', axis=1)
ctff_m_large = ctff_m.loc[ctff_m['size_grp'] == 'large'].copy().drop(columns='size_grp', axis=1)
ctff_m_small = ctff_m.loc[ctff_m['size_grp'] == 'small'].copy().drop(columns='size_grp', axis=1)

In [13]:
# Stand-normalize
ctff_m_rank_all = rank_normalize(ctff_m_all, exclude_cols=['ret_exc_lead1m'])
ctff_m_rank_mega = rank_normalize(ctff_m_mega, exclude_cols=['ret_exc_lead1m'])
ctff_m_rank_large = rank_normalize(ctff_m_large, exclude_cols=['ret_exc_lead1m'])
ctff_m_rank_small = rank_normalize(ctff_m_small, exclude_cols=['ret_exc_lead1m'])

Column:   0%|          | 0/153 [00:00<?, ?it/s]

Column:   0%|          | 0/153 [00:00<?, ?it/s]

Column:   0%|          | 0/153 [00:00<?, ?it/s]

Column:   0%|          | 0/153 [00:00<?, ?it/s]

In [14]:
# Save
ctff_m_rank_all.to_parquet(get_data() / 'ctff' / 'ctff_m_rank_all.pq', compression='brotli')
ctff_m_rank_mega.to_parquet(get_data() / 'ctff' / 'ctff_m_rank_mega.pq', compression='brotli')
ctff_m_rank_large.to_parquet(get_data() / 'ctff' / 'ctff_m_rank_large.pq', compression='brotli')
ctff_m_rank_small.to_parquet(get_data() / 'ctff' / 'ctff_m_rank_small.pq', compression='brotli')

In [15]:
ctff_m_rank_small

age   aliq_at  aliq_mat  ami_126d     at_be    at_gr1  \
id    eom                                                                    
10006 19521231  0.500000 -0.267677  0.314815 -0.479381  0.253086 -0.247475   
      19530131  0.500000 -0.267677  0.327160 -0.469072  0.240741 -0.237374   
      19530228  0.500000 -0.267677  0.402439 -0.458333  0.231707 -0.247475   
      19530331  0.500000 -0.270833  0.375000 -0.489362  0.262500 -0.229167   
      19530430  0.500000 -0.122449  0.386076 -0.479381  0.272152 -0.040816   
...                  ...       ...       ...       ...       ...       ...   
93428 20171130 -0.221251  0.347800  0.008772 -0.414286 -0.036935  0.134921   
      20171231 -0.220395  0.348148  0.005867 -0.451807 -0.045143  0.142151   
      20180131 -0.216043  0.325930  0.015803 -0.472222 -0.070321  0.179144   
93429 20110630 -0.464621 -0.456316 -0.399628 -0.227173 -0.340317 -0.497797   
      20110731 -0.466630 -0.419487 -0.226829 -0.230937 -0.236547 -0.497819   

                   at_me  at_turnover   be_gr1a     be_me  ...  taccruals_ni  \
id    eom                                                  ...                 
10006 19521231  0.489899    -0.348485 -0.216049  0.487654  ...     -0.339506   
      19530131  0.489899    -0.358586 -0.191358  0.500000  ...     -0.339506   
      19530228  0.479798    -0.348485 -0.207317  0.487805  ...     -0.329268   
      19530331  0.489583    -0.364583 -0.212500  0.500000  ...     -0.350000   
      19530430  0.479592    -0.346939 -0.120253  0.487342  ...     -0.322785   
...                  ...          ...       ...       ...  ...           ...   
93428 20171130 -0.287302    -0.086354  0.294843 -0.314774  ...      0.472428   
      20171231 -0.286418    -0.086416  0.295799 -0.308000  ...      0.472497   
      20180131 -0.278846    -0.075431  0.319932 -0.292913  ...      0.398178   
93429 20110630 -0.480198     0.167774 -0.436490 -0.446772  ...      0.062775   
      20110731 -0.477124     0.114035 -0.363636 -0.451794  ...      0.084515   

                tangibility  tax_gr1a  turnover_126d  turnover_var_126d  \
id    eom                                                                 
10006 19521231     0.267677  0.146465       0.388889          -0.489899   
      19530131     0.287879  0.186869       0.318182          -0.479798   
      19530228     0.287879  0.196970       0.419192           0.378788   
      19530331     0.302083  0.229167       0.458333           0.208333   
      19530430     0.295918  0.448980       0.469388           0.081633   
...                     ...       ...            ...                ...   
93428 20171130    -0.134356  0.003727       0.426984           0.485185   
      20171231    -0.136595  0.004983       0.438664           0.467141   
      20180131    -0.129814 -0.147249       0.445513           0.458333   
93429 20110630     0.468137 -0.429438       0.164466           0.380088   
      20110731     0.470838  0.196605       0.157952           0.401961   

                 z_score  zero_trades_126d  zero_trades_21d  zero_trades_252d  \
id    eom                                                                       
10006 19521231  0.000000         -0.308081        -0.419192         -0.328283   
      19530131  0.000000         -0.257576        -0.358586         -0.328283   
      19530228  0.000000         -0.308081        -0.489899         -0.348485   
      19530331  0.000000         -0.343750        -0.447917         -0.375000   
      19530430  0.000000         -0.489796        -0.459184         -0.357143   
...                  ...               ...              ...               ...   
93428 20171130  0.064226         -0.426984        -0.478836         -0.365608   
      20171231  0.072671         -0.438664        -0.458379         -0.381709   
      20180131  0.082324         -0.445513        -0.434829         -0.400641   
93429 20110630  0.426946         -0.164466        -0.128163         -0.406491   
      20110731  0